# 🧠 任务规划智能体 — 完整架构与实现流程> **文档版本:** v0.7  > **日期:** 2026-05-20  > **项目路径:** `d:\Demo\`  > **当前状态:** 5/5 核心模块完成，桌面 exe 已作为主入口，支持创建/导入任务、打卡和动态重规划---## 📌 项目概述### 目标构建一个 AI 智能体，当用户输入**多任务文件**（PDF、Word、TXT/MD、手动打字）时，例如：- “我要七天背 500 个单词”- 数学作业描述 PDF- 多个任务混合输入：背诵、练习、阅读、写作、项目智能体能够：1. **解析文档并提取纯文本**2. **分析每个任务的类型、难度、DDL 和预估耗时**3. **结合节假日、工作日/周末可用时间和手动忙碌时间**4. **生成尽量均匀、不堆叠的每日计划**5. **在桌面便利贴里打卡、创建/导入任务并动态重规划**最终给用户**规划每天要干什么**，例如：> “今天背单词 72 个（约 3h），做数学题 5 道（约 1.5h）。如果新增论文任务，系统会重新分配后续每日安排。”---### 核心用户故事| 用户输入 | 智能体输出 ||----------|------------|| PDF: 数学第三章习题（20 题，2 周内交） | 每天做 2 题，周末集中攻克难题 || 手动输入: 7 天背 500 单词 | 每天背约 72 个，按 DDL 分配 || 同时提交背单词和数学作业 | 根据 DDL、难度和可用时间交替安排 || exe 设置窗口临时新增论文任务 | 合并任务、保留已勾选状态、动态重规划 |

## 🏗️ 整体架构图智能体由 **5 个核心模块** 串联组成，数据从用户输入流经各模块，最终进入每日追踪闭环。```┌──────────────────────────────────────────────────────────────────────────┐│                      任务规划智能体 (Task Planner Agent)                  │├───────────┬───────────┬───────────┬──────────────┬──────────────────────┤│  ① 文档   │  ② 任务   │  ③ 日程   │   ④ 规划     │   ⑤ 输出与追踪        ││  解析器   │  分析器   │  感知器   │   引擎       │                      ││           │           │           │              │                      ││ PDF/Word  │ LLM 分析  │ 节假日API │  约束求解    │  便利贴打卡          ││ TXT/手输  │ 类型/难度 │ 用户可用  │  时间分配    │  三档目标            ││ ↓ text    │ 时长/DDL  │ 忙碌扣除  │  DDL 优先    │  导入任务+重规划     │└───────────┴───────────┴───────────┴──────────────┴──────────────────────┘       ↑                                                         ↓       └──────────────── 用户新增任务 / 打卡反馈 / 重规划 ───────┘```### 数据流详解```mermaidgraph LR    A[上传 PDF/Word/TXT/MD] --> B[① 文档解析]    C[手动输入文本] --> B    B --> D[② 任务分析 LLM]    D --> E[Task JSON 类型/难度/时长/DDL]    E --> F[③ 日程感知 节假日+可用时间]    F --> G[DailySlot 可用时间表]    E --> H[④ 规划引擎]    G --> H    H --> I[PlanResult 每日计划]    I --> J[⑤ 便利贴打卡]    J --> K[导入新任务]    K --> D    J --> L[SQLite tracker.db]    L --> J```### 入口分工| 入口 | 文件 | 用途 ||------|------|------|| 桌面 exe 主入口 | `dist/TaskPlannerDesktop.exe` | 推荐使用：桌面便签、创建/导入任务、打卡、重规划 || 桌面开发入口 | `desktop_app/main.py` | 与 exe 同功能，便于调试 || Streamlit 配置 App | `app.py` | 可选开发/调试入口：解析 → 分析 → 日程 → 规划 || Streamlit 便利贴 App | `tracker_app.py` | 可选开发/调试入口：Web 版打卡和重规划 |桌面 exe 已迁移 Streamlit 的核心创建流程：用户不需要先进入网页端创建任务，可以直接在 exe 的设置窗口里输入任务或选择文件。

---## 📄 模块 ① — 文档解析器（✅ 已完成）### 职责将用户上传的各种格式文件**统一转换为纯文本**，供下游 LLM 分析。### 支持格式| 格式 | 解析库 | 说明 ||------|--------|------|| **PDF** | PyMuPDF (fitz) | 逐页提取文本，检测加密 PDF || **Word .docx** | python-docx | 提取段落 + 表格中的文本 || **TXT / MD** | 原生 Python | 直接读取文件 || **手动输入** | 无需解析 | 直接传入字符串 |### 数据结构```python@dataclassclass ParsedDocument:    source: str    file_type: str    raw_text: str    page_count: int | None    metadata: dict```### 当前文件结构```doc_parser/├── __init__.py├── models.py        # ParsedDocument├── parser.py        # 统一入口 + 路由├── pdf_parser.py    # PDF 解析├── word_parser.py   # Word 解析├── text_parser.py   # 纯文本解析└── i18n.py          # 中英文翻译（UI 专用）```### 测试结果6/6 通过：手动输入、TXT 文件、不支持类型、文件不存在、参数冲突、空参数。

---## 🧠 模块 ② — 任务分析器（✅ 已完成）### 职责将模块 ① 提取的纯文本，通过 LLM **结构化分析**，输出标准任务对象。### 核心能力- OpenAI SDK 兼容调用，默认支持 DeepSeek，也支持 OpenAI- `response_format={"type":"json_object"}` 强制 JSON 输出- Prompt 内置效率先验：背单词、数学题、阅读、写作、项目任务等- 自动识别多任务并拆分- 输出 `TaskAnalysisResult`，其中包含 `tasks`、`warnings`、`raw_response` 等信息### LLM 返回结构示例```json{  "tasks": [    {      "id": "task_001",      "description": "背诵 500 个英语单词",      "task_type": "memorize",      "total_amount": 500,      "unit": "words",      "difficulty": 3,      "estimated_hours": 25.0,      "unit_efficiency": 20,      "deadline": "2026-05-26",      "suggested_daily_hours": 3.5,      "confidence": 0.8,      "notes": "建议分散复习"    }  ],  "warnings": []}```### 当前文件结构```task_analyzer/├── __init__.py├── models.py        # Task / TaskAnalysisResult├── analyzer.py      # 核心分析编排├── llm_client.py    # DeepSeek / OpenAI 客户端└── prompts.py       # Prompt 模板 + 效率先验```### 测试结果6/6 通过：模型序列化、计算属性、聚合结果、Prompt 构建、客户端配置、错误处理。

---## 📅 模块 ③ — 日程感知器（✅ 已完成）### 职责根据用户设定和节假日信息生成每日可用时间表。### 数据来源| 数据源 | 获取方式 | 当前状态 ||--------|----------|----------|| 中国法定节假日 | `timor.tech/api/holiday` | ✅ 已接入，按年缓存 || 调休工作日 | 同上 | ✅ 已识别 || 用户基础可用时间 | 工作日/周末/节假日小时数 | ✅ 已支持 || 手动忙碌时间 | `YYYY-MM-DD=hours` | ✅ 已支持 || OAuth 日历 | Google/Outlook | P2，可后续扩展 |### 核心模型```python@dataclassclass UserSettings:    workday_hours: float    weekend_hours: float    holiday_hours: float    manual_busy: dict[str, float]@dataclassclass DailySlot:    date: date    day_of_week: str    available_hours: float    base_hours: float    manual_busy_hours: float    is_holiday: bool    is_workday: bool```### 核心逻辑1. 判断日期类型：工作日、周末、法定节假日、调休工作日2. 读取用户设置的基础可用小时数3. 扣除手动忙碌时间4. 输出 `dict[date, DailySlot]`### 测试结果7/7 通过：数据模型、基础小时计算、节假日 API、日程生成、忙碌扣除、非法日期、序列化。

,---## ⚙️ 模块 ④ — 规划引擎（✅ 已完成）### 职责在多任务、多 DDL、有限可用时间的约束下，生成每日任务分配。### 已实现策略- **DDL 优先**：越早截止越优先- **最低保障**：先计算每个任务为了不超期需要的最低每日投入- **比例分配**：剩余时间按剩余工作量比例分配，避免单任务独占- **每日缓冲**：默认预留 10% 空余时间- **单任务上限**：单个任务每天最多 4h，避免疲劳- **均匀分配**：默认 `max_tasks_per_day=2`，普通任务轮换安排，减少同一天任务堆叠- **紧急突破**：DDL 3 天内的任务视为紧急任务，优先安排；必要时允许超过每日任务数量限制- **不可行告警**：总需求过高或 DDL 前可用时间不足时生成 warnings### 核心模型```python@dataclassclass TaskAllocation:    task_id: str    description: str    amount: float    unit: str    hours: float@dataclassclass DailyPlan:    date: date    available_hours: float    allocations: list[TaskAllocation]@dataclassclass PlanResult:    days: list[DailyPlan]    progress: dict[str, TaskProgress]    warnings: list[str]```### 简化算法```pythonfor day in slots:    usable = day.available_hours * (1 - buffer_ratio)    active = tasks_with_remaining_work()    urgent = tasks_due_within_3_days(active)    normal = rotate_normal_tasks(active, max_tasks_per_day - len(urgent))    today_tasks = urgent + normal    min_allocations = allocate_minimum_required(today_tasks)    extra_allocations = allocate_remaining_time_proportionally(today_tasks)    update_progress(min_allocations + extra_allocations)```### 测试结果7/7 通过：规划模型、基础规划、DDL 优先、缓冲比例、单任务完成、不可行检测、结果序列化。

---## 📊 模块 ⑤ — 输出与追踪（✅ 已完成）### 职责将计划变成用户每天真正能用的“便利贴”：每天打开即可看到问候、目标、打卡选项和进度回顾。### 已实现能力- **无计划自动创建**：`tracker_app.py` 检测不到计划时，直接显示简化配置界面- **上传/输入一体化**：支持上传 PDF/Word/TXT/MD，也支持直接输入任务描述- **内嵌完整链路**：解析 → AI 分析 → 日程设置 → 生成计划 → 保存快照- **双阶段打卡**：阶段一问候、鸡汤、昨日回顾；阶段二任务清单- **三档目标**：最低 / 理想 / 挑战，同一任务三档互斥选择- **导入新任务**：在任务列表或回顾页导入文本/文件- **动态重规划**：新任务合并后重新分配全部任务，尽量平均，保留当天已勾选状态- **SQLite 持久化**：`tracker.db` 保存计划快照、打卡状态、历史记录、用户设置### tracker_app 无计划流程```mermaidgraph TD    A[打开 tracker_app.py] --> B{是否存在 plan_snapshot?}    B -->|否| C[显示简化配置界面]    C --> D[上传文件或输入文本]    D --> E[文档解析]    E --> F[LLM 分析任务]    F --> G[设置可用时间和日期范围]    G --> H[规划引擎生成计划]    H --> I[保存 tracker.db]    I --> J[进入每日打卡]    B -->|是| J```### 导入新任务流程```pythondef import_new_task(raw_text):    new_tasks = TaskAnalyzer(...).analyze(raw_text).tasks    merged_tasks = merge_tasks(existing_tasks, new_tasks)    new_plan = PlanningEngine(merged_tasks, existing_slots, max_tasks_per_day=2).plan()    save_plan_snapshot(merged_tasks, existing_slots, new_plan)    reset_today_state_after_replan(new_plan, preserve_selected_tiers=True)```### 测试结果8/8 通过：任务档位、状态属性、挑战判定、SQLite CRUD、今日状态、双阶段流程、结算、鸡汤/笑话。

---## 🛠️ 技术栈与当前项目结构### 技术栈| 层次 | 当前方案 ||------|----------|| UI | Streamlit || 文档解析 | PyMuPDF、python-docx、原生文本读取 || LLM 调用 | OpenAI SDK，兼容 DeepSeek / OpenAI || 节假日 | timor.tech API + requests || 规划算法 | 自研约束分配 + DDL 优先 + 均匀轮换 || 持久化 | SQLite (`tracker.db`) || 测试 | Python 脚本测试，当前 34/34 通过 |### 当前项目结构```d:\Demo├── app.py                    # 完整配置 App├── tracker_app.py            # 每日便利贴 App：无计划创建 + 导入重规划├── requirements.txt├── test_parser.py            # 6/6├── test_analyzer.py          # 6/6├── test_schedule.py          # 7/7├── test_planner.py           # 7/7├── test_tracker.py           # 8/8├── README.md├── 任务规划智能体_架构与流程.ipynb├── doc_parser/├── task_analyzer/├── schedule_engine/├── planning_engine/└── daily_tracker/```### 运行方式```powershellcd d:\Demouv venvuv pip install -r requirements.txt# 可选：完整配置入口.venv\Scripts\python.exe -m streamlit run app.py# 日常入口；没有计划时也可以直接创建.venv\Scripts\python.exe -m streamlit run tracker_app.py```### 开发状态| 阶段 | 内容 | 状态 ||------|------|------|| 1 | 文档解析 | ✅ 完成 || 2 | 任务分析 | ✅ 完成 || 3 | 日程感知 | ✅ 完成 || 4 | 规划引擎 | ✅ 完成 || 5 | 输出与追踪 | ✅ 完成 |---## ⚠️ 已解决的关键问题| 问题 | 处理方式 ||------|----------|| 外部管理 Python 环境导致 pip 安装失败 | 使用 `uv venv` + `uv pip install` || parser 循环导入 | 将 `ParsedDocument` 移到 `doc_parser/models.py` || `.txt` 文件与 raw text 路由混淆 | 统一由 `parse_document` 判断参数 || 节假日 API 不稳定 | 按年缓存，失败时按普通日历降级 || SQLite 连接占用测试 DB | 显式关闭连接 || tracker 无计划时只能提示去主 App | 内嵌创建模式，直接完成计划生成 || 新增任务后计划不更新 | 导入新任务后合并并重规划，保留当日选择 |---> 📌 **当前结论：** 项目五个模块全部完成，`tracker_app.py` 已具备日常独立入口能力：无计划可创建，有计划可打卡，新任务可导入并动态重规划。